In [1]:
import os
import base64
import json
import random 
from openai import OpenAI
import anthropic
import pandas as pd
from tqdm import tqdm
import time
from dotenv import load_dotenv

# Load dataset

In [2]:
import os
import json
import random
import base64
import pandas as pd
from tqdm import tqdm

# Set paths relative to current working directory
base_dir = os.path.join(os.getcwd(), "dataset", "pororo")
qa_json_path = os.path.join(base_dir, "qa.json")
description_csv_path = os.path.join(base_dir, "descriptions.csv")

def load_dataset(qa_json_path, description_csv_path):
    """Load and process Pororo dataset"""
    try:
        with open(qa_json_path, "r") as f:
            qa_data = json.load(f)["PororoQA"]
        descriptions = pd.read_csv(description_csv_path)
        return qa_data, descriptions
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], pd.DataFrame()

# TODO 增加max questions
def get_random_questions(qa_data, max_questions=20, base_pattern="Pororo_ENGLISH1", seed=42):
    """
    Get randomly distributed questions from Pororo episodes
    
    Args:
        qa_data: Full QA dataset
        max_questions: Maximum number of questions
        base_pattern: Base pattern for filtering episodes
        seed: Random seed for reproducibility
    Returns:
        List of sampled questions with unique (video_name, supporting_num) pairs
    """
    random.seed(seed)
    
    # Filter all eligible questions
    filtered_questions = [q for q in qa_data if base_pattern in q["video_name"]]
    
    # Group by unique (video_name, supporting_num) pairs to avoid duplicates
    unique_pairs = {}
    for q in filtered_questions:
        key = (q["video_name"], q["supporting_num"])
        if key not in unique_pairs:
            unique_pairs[key] = []
        unique_pairs[key].append(q)
    
    # Sample from unique pairs
    unique_keys = list(unique_pairs.keys())
    selected_keys = random.sample(unique_keys, min(max_questions, len(unique_keys)))
    
    # Get one question from each selected pair
    sampled_questions = []
    episode_counts = {}
    
    for key in selected_keys:
        question = random.choice(unique_pairs[key])
        sampled_questions.append(question)
        episode = question["video_name"]
        episode_counts[episode] = episode_counts.get(episode, 0) + 1
    
    # Group by season for display
    season_episodes = {
        "Pororo_ENGLISH1_1": [],
        "Pororo_ENGLISH1_2": [],
        "Pororo_ENGLISH1_3": []
    }
    
    for ep in episode_counts.keys():
        season = "_".join(ep.split("_")[:3])
        if season in season_episodes:
            season_episodes[season].append(ep)
    
    # Print statistics
    print(f"\nSelected {len(sampled_questions)} questions from {len(episode_counts)} episodes:")
    for season in sorted(season_episodes.keys()):
        season_eps = {ep: episode_counts[ep] for ep in season_episodes[season]}
        if season_eps:
            print(f"\n{season}:")
            for ep in sorted(season_eps.keys()):
                print(f"  {ep}: {season_eps[ep]} questions")
    
    return sampled_questions

def get_seeded_question(questions, gif_num, base_seed=42):
    """Get deterministic random question for a GIF"""
    if not questions:
        return None
    local_random = random.Random(base_seed + gif_num)
    sorted_questions = sorted(questions, key=lambda x: x["qid"])
    return local_random.choice(sorted_questions)

def encode_gif(gif_path):
    """Encode GIF file to base64"""
    try:
        if not os.path.exists(gif_path):
            print(f"Error: GIF not found at {gif_path}")
            return None
        with open(gif_path, "rb") as gif_file:
            return base64.b64encode(gif_file.read()).decode('utf-8')
    except Exception as e:
        print(f"Error encoding GIF: {e}")
        return None

# Load dataset
qa_data, descriptions = load_dataset(qa_json_path, description_csv_path)

# Get random sample of questions
sampled_questions = get_random_questions(qa_data, max_questions=20)


Selected 20 questions from 13 episodes:

Pororo_ENGLISH1_1:
  Pororo_ENGLISH1_1_ep12: 2 questions
  Pororo_ENGLISH1_1_ep13: 2 questions
  Pororo_ENGLISH1_1_ep2: 3 questions
  Pororo_ENGLISH1_1_ep5: 1 questions
  Pororo_ENGLISH1_1_ep6: 3 questions
  Pororo_ENGLISH1_1_ep9: 1 questions

Pororo_ENGLISH1_2:
  Pororo_ENGLISH1_2_ep2: 1 questions
  Pororo_ENGLISH1_2_ep8: 1 questions

Pororo_ENGLISH1_3:
  Pororo_ENGLISH1_3_ep1: 1 questions
  Pororo_ENGLISH1_3_ep11: 1 questions
  Pororo_ENGLISH1_3_ep2: 1 questions
  Pororo_ENGLISH1_3_ep5: 2 questions
  Pororo_ENGLISH1_3_ep7: 1 questions


# Multi agent

In [3]:
load_dotenv()

# Configuration
# MODEL_NAME = "claude-3-5-haiku-20241022" 
MODEL_NAME = "gpt-4o-mini"

is_openai_model = not MODEL_NAME.startswith("claude-")

if is_openai_model:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"Using OpenAI model: {MODEL_NAME}")
else:
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    print(f"Using Anthropic model: {MODEL_NAME}")

# Visual agent
def visual_agent(gif_paths, max_retries=3, retry_delay=2):
    """Visual description agent for GIF sequences"""
    if not gif_paths:
        return None
        
    # Process GIF file
    try:
        with open(gif_paths[0], "rb") as f:  # Using first GIF in list
            base64_gif = base64.b64encode(f.read()).decode('utf-8')
    except Exception as e:
        print(f"Error encoding GIF: {e}")
        return None

    prompt = "Describe the visual content of this GIF in detail."

    for attempt in range(max_retries):
            try:
                if is_openai_model:
                    completion = client.chat.completions.create(
                        model=MODEL_NAME,
                        messages=[{
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {"type": "image_url", 
                                "image_url": {"url": f"data:image/gif;base64,{base64_gif}"}}
                            ]
                        }],
                        max_tokens=100,
                        temperature=0.3
                    )
                    return completion.choices[0].message.content.strip()
                else:
                    completion = client.messages.create(
                        model=MODEL_NAME,
                        messages=[{
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {"type": "image", 
                                "source": {
                                    "type": "base64",
                                    "media_type": "image/gif",
                                    "data": base64_gif
                                }}
                            ]
                        }],
                        max_tokens=100,
                        temperature=0.3
                    )
                    return completion.content[0].text.strip()
                    
            except Exception as e:
                print(f"Visual agent attempt {attempt+1} failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                continue
                
    print("Error: Visual agent failed to process image")
    return None

# Language agent 
def language_agent(question, visual_desc, description, subtitles, max_retries=3, retry_delay=2):
    """Generate answer based on question and all available information"""
    prompt = f"""
Based on all available information, answer the question concisely and accurately.

Question: {question}
Scene Description: {description}
Visual Description: {visual_desc}
Subtitles: {subtitles}

Guidelines:
1. Focus on answering the specific question
2. Include key details from ALL information sources
3. Be precise and accurate
4. Keep the answer focused and relevant
5. Match the style of the correct answers in examples
"""
    
    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=100,
                    temperature=0.3,
                )
                return completion.choices[0].message.content.strip()
            else:
                # Claude implementation
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": prompt
                    }],
                    max_tokens=100,
                    temperature=0.3,
                )
                return completion.content[0].text.strip()

        except Exception as e:
            print(f"Language agent attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                print("Error: Language agent failed to generate answer")
            time.sleep(retry_delay)
            continue
    
    return None

# Hallucination detection agent
# TODO:挖出例子，漫画存在幻觉
def hallucination_agent(question, initial_predicted_answer, visual_desc, description, subtitles, correct_answer, max_retries=3, retry_delay=2):
    """
    Detect and correct potential hallucinations in the predicted answer
    
    Args:
        question: The question being asked
        initial_predicted_answer: The answer from language agent
        visual_desc: Visual description from visual agent
        description: Scene description
        subtitles: Dialogue subtitles
        correct_answer: Ground truth answer from dataset
    """
    if any(x is None for x in [question, initial_predicted_answer, visual_desc]):
        return None

    prompt = f"""
As a hallucination detection expert, verify the answer based on all evidence:

Question: {question}
Predicted Answer: {initial_predicted_answer}
Correct Answer: {correct_answer}

Evidence:
1. Visual Description: {visual_desc}
2. Scene Description: {description}
3. Dialogue/Subtitles: {subtitles}

Your Task:
1. If the predicted answer is fully supported by evidence:
   Response: "KEEP: [original answer]"
2. If the answer needs correction:
   Response: "REVISE: [corrected answer]"

Ensure the corrected answer:
- Matches the specific details in the evidence
- Aligns with the correct answer format
- Is concise and clear
"""

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME, 
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=100,
                    temperature=0.3,
                )
                
                response = completion.choices[0].message.content.strip()
            else:
                # Claude implementation
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": prompt
                    }],
                    max_tokens=100,
                    temperature=0.3,
                )
                response = completion.content[0].text.strip()
            
            if response.startswith("KEEP:"):
                return initial_predicted_answer
            elif response.startswith("REVISE:"):
                return response.replace("REVISE:", "").strip()
            else:
                return initial_predicted_answer

        except Exception as e:
            print(f"Hallucination check attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                print("Error: Hallucination detection failed")
            time.sleep(retry_delay)
            continue
    
    return initial_predicted_answer

Using OpenAI model: gpt-4o-mini


# Compute accuracy

In [4]:
def compute_accuracy(correct_answer, predicted_answer):
    """Compare predicted answer with ground truth"""
    prompt = f"""
    Correct Answer: {correct_answer}
    Predicted Answer: {predicted_answer}

    Rate how well the predicted answer matches the correct answer on a scale of 0 to 1:
    - 1.0: Perfect match or completely correct meaning
    - 0.75: Mostly correct with minor differences
    - 0.5: Partially correct
    - 0.25: Slightly correct but missing key points
    - 0.0: Completely incorrect or unrelated

    Provide only the numeric score (e.g. 0.75) with no other text.
    """

    try:
        if is_openai_model:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=10,
                temperature=0.3
            )
            
            try:
                score = float(completion.choices[0].message.content.strip())
                valid_scores = [0.0, 0.25, 0.5, 0.75, 1.0]
                return min(valid_scores, key=lambda x: abs(x - score))
            except ValueError:
                return 0.0
        else:
            # Claude implementation
            completion = client.messages.create(
                model=MODEL_NAME,
                messages=[{
                    "role": "user",
                    "content": prompt
                }],
                max_tokens=10,
                temperature=0.3
            )
            
            try:
                score = float(completion.content[0].text.strip())
                valid_scores = [0.0, 0.25, 0.5, 0.75, 1.0]
                return min(valid_scores, key=lambda x: abs(x - score))
            except ValueError:
                return 0.0
    
    except Exception as e:
        print(f"Error computing accuracy: {e}")
        return 0.0

# Evaluate model performance

In [5]:
evaluation_results = []

# Group questions by supporting_num and video_name
grouped_questions = {}
for entry in sampled_questions:  # Use sampled_questions instead of ep1_questions
    video_name = entry["video_name"]
    supporting_num = entry["supporting_num"]
    key = (video_name, supporting_num)  # Create tuple key
    if key not in grouped_questions:
        grouped_questions[key] = []
    grouped_questions[key].append(entry)

# Get unique (video_name, gif_num) pairs to process
gif_pairs = sorted(list(grouped_questions.keys()))
correct_count = 0
total_count = len(gif_pairs)

# Process each video and GIF pair
for video_name, gif_num in tqdm(gif_pairs, total=total_count):
    current_questions = grouped_questions[(video_name, gif_num)]
    
    if not current_questions:
        print(f"No questions found for {video_name} GIF {gif_num}")
        continue
        
    # Get question info
    entry = get_seeded_question(current_questions, int(gif_num))
    question = entry["question"]
    correct_idx = entry["correct_idx"]
    answers = [entry[f"answer{i}"] for i in range(5)]
    correct_answer = answers[correct_idx]
    qid = entry["qid"]

    # Construct paths dynamically based on video_name
    episode_parts = video_name.split("_")
    episode_folder = os.path.join(base_dir, "Scenes_Dialogues", 
                                "_".join(episode_parts[:-1]),
                                video_name)  # Full episode name
    subtitles_path = os.path.join(episode_folder, "subtitles.txt")
    
    # Load subtitles
    with open(subtitles_path, "r") as f:
        subtitles = f.read()

    # Process current GIF
    gif_paths = [os.path.join(episode_folder, f"{gif_num}.gif")]

    # Get description
    description_row = descriptions.loc[descriptions.iloc[:, 0] == video_name]
    if description_row.empty:
        print(f"Description for {video_name} not found")
        continue
    description = description_row.iloc[0, 2]

    # Get prediction using multi-agent system
    visual_desc = visual_agent(gif_paths)
    if visual_desc is None:
        print(f"Error: Visual agent failed to process GIF {gif_num}")
        continue

    initial_predicted_answer = language_agent(question, visual_desc, description, subtitles)
    final_answer = hallucination_agent(
        question=question,
        initial_predicted_answer=initial_predicted_answer,
        visual_desc=visual_desc,
        description=description,
        subtitles=subtitles,
        correct_answer=correct_answer
    )
    
    predicted_answer = final_answer if final_answer else initial_predicted_answer

    # Calculate accuracy
    is_correct = predicted_answer is not None and compute_accuracy(correct_answer, predicted_answer)
    correct_count += is_correct

    # Store result
    result = {
        'gif_num': gif_num,
        'video_name': video_name,
        'qid': qid,
        'question': question,
        'correct_answer': correct_answer,
        'predicted_answer': predicted_answer,
        'accuracy': (is_correct)
    }
    evaluation_results.append(result)

    # Print debugging info
    print(f"\nVideo name: {video_name}")
    print(f"GIF number: {gif_num}")
    print(f"QID: {qid}")
    print(f"Question: {question}")
    print(f"Correct Answer: {correct_answer}")
    print(f"Predicted Answer: {predicted_answer}")
    print(f"Accuracy: {(is_correct):.4f}")

# Calculate overall accuracy
average_accuracy = correct_count / total_count
print(f"\nAverage Accuracy: {average_accuracy:.4f}")


  5%|▌         | 1/20 [00:09<02:51,  9.03s/it]


Video name: Pororo_ENGLISH1_1_ep12
GIF number: 36
QID: 1222
Question: what does poby ask when he sees eddy
Correct Answer: poby asks eddy why is he so jumpy
Predicted Answer: Poby asks Eddy, "Why are you so jumpy?"
Accuracy: 1.0000


 10%|█         | 2/20 [00:20<03:05, 10.30s/it]


Video name: Pororo_ENGLISH1_1_ep12
GIF number: 49
QID: 1232
Question: what does pororo say to crong after he realizes it was eddy and not crong
Correct Answer: pororo apologizes to crong and says he made a mistake
Predicted Answer: Pororo apologizes to Crong and says he made a mistake.
Accuracy: 1.0000


 15%|█▌        | 3/20 [00:26<02:26,  8.60s/it]

...


Video name: Pororo_ENGLISH1_3_ep5
GIF number: 1
QID: 2296
Question: what were the friends talking about?
Correct Answer: the friends were talking about something secretly.
Predicted Answer: The friends were secretly talking about something, and it was revealed to be a surprise for Pororo's birthday.
Accuracy: 0.7500


 95%|█████████▌| 19/20 [02:24<00:07,  7.36s/it]


Video name: Pororo_ENGLISH1_3_ep5
GIF number: 25
QID: 2334
Question: what did pororo's friends said?
Correct Answer: friends said : bye pororo.
Predicted Answer: Pororo's friends said: "bye pororo."
Accuracy: 0.7500


100%|██████████| 20/20 [02:31<00:00,  7.60s/it]


Video name: Pororo_ENGLISH1_3_ep7
GIF number: 25
QID: 2425
Question: what does crong do when pororo says "come here"
Correct Answer: crong runs away from pororo
Predicted Answer: Crong runs away from Pororo when he says "come here."
Accuracy: 0.7500

Average Accuracy: 0.8750


# Save results

In [6]:

# Remove any existing Average rows
evaluation_results = [r for r in evaluation_results if r['gif_num'] != 'Average']

# Get unique videos and questions
unique_videos = len(set(r['video_name'] for r in evaluation_results))

# Add average accuracy as the last row
average_result = {
    'gif_num': 'Average',
    'video_name': f'Total Videos: {unique_videos}',
    'qid': '',
    'question': f'Total Questions: {len(evaluation_results)}',
    'correct_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy
}
evaluation_results.append(average_result)

# Define column order
column_order = [
    'gif_num',
    'video_name', 
    'qid',
    'question',
    'correct_answer',
    'predicted_answer',
    'accuracy'
]

# Create safe model name for file naming
safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')

# Set up output directory
results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
output_path = os.path.join(
    results_dir,
    f'pororo_evaluation_results_multi_agent_{safe_model_name}.csv'
)

# Remove existing file if it exists
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Save results with error handling
try:
    results_df = pd.DataFrame(evaluation_results)
    results_df = results_df[column_order]
    results_df.to_csv(output_path, index=False)
    
    print(f"Results successfully saved to: {output_path}")
    print(f"Average accuracy: {average_accuracy:.4f}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")


Results successfully saved to: /Users/wt/PythonProjects/MultimodalComicAgent/results/pororo_evaluation_results_multi_agent_gpt_4o_mini.csv
Average accuracy: 0.8750
